In [ ]:
import sys
from itertools import product
from math import pi
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, IFrame, display

for candidate in (
    Path.cwd() / "Privacy-Scenario",
    Path.cwd(),
    Path.home() / "Desktop" / "COORDINATE-2.0" / "Privacy-Scenario",
):
    if (candidate / "run_privacy_experiments.py").exists():
        SCENARIO_ROOT = candidate
        break
else:
    raise FileNotFoundError("Privacy-Scenario/run_privacy_experiments.py not found.")

sys.path.insert(0, str(SCENARIO_ROOT))
import run_privacy_experiments as scenario

PLOTS_DIR = SCENARIO_ROOT / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
summary_candidates = [
    scenario.RESULTS_ROOT / "privacy_summary.csv",
    scenario.RESULTS_ROOT / "smoke" / "privacy_summary.csv",
]
summary_path = next((path for path in summary_candidates if path.exists()), None)
if summary_path is None:
    raise FileNotFoundError("Run the privacy experiment campaign before generating plots.")

summary = pd.read_csv(summary_path)
required_columns = {
    "topology_label",
    "workload",
    "technique",
    "architecture_label",
    "avg_latency_s",
    "avg_throughput_tasks_per_hour",
    "avg_success_rate",
    "avg_expiration_rate",
    "avg_privacy_processing_seconds",
    "avg_privacy_added_delay_seconds",
    "avg_adversary_success_rate",
    "avg_expected_distance_error",
}
missing_columns = sorted(required_columns - set(summary.columns))
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

topology_order = [name for name in scenario.TOPOLOGIES if name in set(summary["topology_label"])]
technique_order = [name for name in scenario.TECHNIQUES if name in set(summary["technique"])]
workload_order = [name for name in scenario.WORKLOADS if name in set(summary["workload"])]
arch_order = [name for name in scenario.ARCHITECTURES if name in set(summary["architecture_label"])]
technique_labels = {
    "none": "None",
    "encryption": "Encryption",
    "obfuscation": "Obfuscation",
    "combined": "Combined",
}
colors = {
    "FD": "#666666",
    "SD": "#999999",
    "CE": "#b3b3b3",
}
technique_colors = {
    "none": "#A8DADC",
    "encryption": "#F6BD60",
    "obfuscation": "#B8E0D2",
    "combined": "#D7BDE2",
}

TABLES_DIR = SCENARIO_ROOT / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

table_specs = [
    (
        "performance",
        "Impact of coordination mechanisms on system latency and throughput under different topology, workload, and privacy conditions.",
        "tab:privacy_performance",
        ("avg_latency_s", "Avg. Latency (sec.)", 1.0, "min"),
        ("avg_throughput_tasks_per_hour", "Throughput (tasks/hour)", 1.0, "max"),
    ),
    (
        "effectiveness",
        "Privacy achieved by the coordination mechanisms under different topology, workload, and privacy conditions.",
        "tab:privacy_effectiveness",
        ("avg_adversary_success_rate", "Adversary Success Rate (%)", 100.0, "min"),
        ("avg_expected_distance_error", "Expected Distance Error (hops)", 1.0, "max"),
    ),
    (
        "overhead",
        "Overhead introduced by the privacy techniques under different topology, workload, and coordination conditions.",
        "tab:privacy_overhead",
        ("avg_privacy_added_delay_seconds", "Added Delay (sec.)", 1.0, "min"),
        ("avg_privacy_processing_seconds", "Processing Time (ms)", 1000.0, "min"),
    ),
]

def metric_values(subset, metric, scale):
    values = []
    for arch in arch_order:
        row = subset[subset["architecture_label"] == arch]
        if row.empty:
            raise ValueError(f"Missing result for {arch}.")
        value = pd.to_numeric(row[metric], errors="coerce").iloc[0]
        values.append(float(value) * scale if pd.notna(value) else float("nan"))
    return values

def latex_cells(values, direction):
    valid = [value for value in values if pd.notna(value)]
    best = (min(valid) if direction == "min" else max(valid)) if valid else None
    return [
        "--" if pd.isna(value) else (f"\\cellcolor[gray]{{0.70}}{value:.2f}" if value == best else f"{value:.2f}")
        for value in values
    ]

def html_cells(values, direction):
    valid = [value for value in values if pd.notna(value)]
    best = (min(valid) if direction == "min" else max(valid)) if valid else None
    return [
        "--" if pd.isna(value) else (f"<strong>{value:.2f}</strong>" if value == best else f"{value:.2f}")
        for value in values
    ]

for table_name, caption, label, first_metric, second_metric in table_specs:
    latex_rows = []
    html_rows = []
    for topology_idx, topology in enumerate(topology_order):
        for workload_idx, workload in enumerate(workload_order):
            for technique_idx, technique in enumerate(technique_order):
                subset = summary[
                    (summary["topology_label"] == topology)
                    & (summary["workload"] == workload)
                    & (summary["technique"] == technique)
                ]
                first = metric_values(subset, first_metric[0], first_metric[2])
                second = metric_values(subset, second_metric[0], second_metric[2])
                topology_cell = f"\\multirow{{{len(workload_order) * len(technique_order)}}}{{*}}{{{topology.title()}}}" if workload_idx == 0 and technique_idx == 0 else ""
                workload_cell = f"\\multirow{{{len(technique_order)}}}{{*}}{{{workload.title()}}}" if technique_idx == 0 else ""
                latex_rows.append(
                    " & ".join([topology_cell, workload_cell, technique_labels[technique], *latex_cells(first, first_metric[3]), *latex_cells(second, second_metric[3])]) + " " + chr(92) * 2
                )
                html_topology = topology.title() if workload_idx == 0 and technique_idx == 0 else ""
                html_workload = workload.title() if technique_idx == 0 else ""
                html_rows.append(
                    "<tr>"
                    + f"<td>{html_topology}</td><td>{html_workload}</td><td>{technique_labels[technique]}</td>"
                    + "".join(f"<td>{value}</td>" for value in html_cells(first, first_metric[3]))
                    + "".join(f"<td>{value}</td>" for value in html_cells(second, second_metric[3]))
                    + "</tr>"
                )
            if workload_idx < len(workload_order) - 1:
                latex_rows.append("\\cmidrule(lr){2-9}")
        if topology_idx < len(topology_order) - 1:
            latex_rows.append("\\midrule")

    row_break = chr(92) * 2
    first_latex_label = first_metric[1].replace("%", "\\%")
    second_latex_label = second_metric[1].replace("%", "\\%")
    latex_table = "\n".join([
        "\\begin{table*}[!t]",
        "\\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        "\\setlength{\\tabcolsep}{6pt}",
        "\\begin{tabular}{lllccc ccc}",
        "\\toprule",
        "\\multirow{2}{*}{\\textbf{Topology}} & \\multirow{2}{*}{\\textbf{Workload}} & \\multirow{2}{*}{\\textbf{Privacy}}",
        f"& \\multicolumn{{3}}{{c}}{{\\textbf{{{first_latex_label}}}}} & \\multicolumn{{3}}{{c}}{{\\textbf{{{second_latex_label}}}}} {row_break}",
        "\\cmidrule(lr){4-6}",
        "\\cmidrule(lr){7-9}",
        f"& & & \\textbf{{CE}} & \\textbf{{SD}} & \\textbf{{FD}} & \\textbf{{CE}} & \\textbf{{SD}} & \\textbf{{FD}} {row_break}",
        "\\midrule",
        *latex_rows,
        "\\bottomrule",
        "\\end{tabular}",
        "\\end{table*}",
        "",
    ])
    table_path = TABLES_DIR / f"privacy_{table_name}_summary.tex"
    table_path.write_text(latex_table, encoding="utf-8")
    table_html = f"""
<h3>{caption}</h3>
<table style="border-collapse:collapse; font-size:12px; margin-bottom:16px; text-align:right;">
  <thead><tr>
    <th rowspan="2" style="padding:4px 8px; text-align:left; border-bottom:1px solid #222;">Topology</th>
    <th rowspan="2" style="padding:4px 8px; text-align:left; border-bottom:1px solid #222;">Workload</th>
    <th rowspan="2" style="padding:4px 8px; text-align:left; border-bottom:1px solid #222;">Privacy</th>
    <th colspan="3" style="padding:4px 8px; text-align:center; border-bottom:1px solid #222;">{first_metric[1]}</th>
    <th colspan="3" style="padding:4px 8px; text-align:center; border-bottom:1px solid #222;">{second_metric[1]}</th>
  </tr><tr>
    <th>CE</th><th>SD</th><th>FD</th><th>CE</th><th>SD</th><th>FD</th>
  </tr></thead><tbody>{''.join(html_rows)}</tbody>
</table>
<p><a href="tables/{table_path.name}">{table_path.name}</a></p>
"""
    display(HTML(table_html))

plot_specs = [
    ("avg_latency_s", "Average Latency (sec.)", "latency_by_technique.pdf"),
    ("avg_throughput_tasks_per_hour", "Throughput (tasks/hour)", "throughput_by_technique.pdf"),
    ("avg_privacy_added_delay_seconds", "Privacy Overhead (sec.)", "privacy_overhead_by_technique.pdf"),
    ("avg_adversary_success_rate", "Adversary's Success Rate", "adversary_success_rate_by_technique.pdf"),
    ("avg_expected_distance_error", "Expected Distance Error (hops)", "expected_distance_error_by_technique.pdf"),
]

for plot_idx, (metric, ylabel, filename) in enumerate(plot_specs):
    fig, axes = plt.subplots(
        len(topology_order),
        len(workload_order),
        figsize=(3.0 * len(workload_order), 2.0 * len(topology_order)),
        sharey=False,
        squeeze=False,
    )

    for topology, workload in product(topology_order, workload_order):
        topology_idx = topology_order.index(topology)
        workload_idx = workload_order.index(workload)
        ax = axes[topology_idx][workload_idx]
        subset = summary[
            (summary["topology_label"] == topology)
            & (summary["workload"] == workload)
        ].copy()
        x = range(len(technique_order))
        width = 0.23
        ax.set_axisbelow(True)
        ax.grid(axis="y", linestyle=":", linewidth=0.5)

        for idx, arch in enumerate(arch_order):
            values = []
            for technique in technique_order:
                row = subset[
                    (subset["technique"] == technique)
                    & (subset["architecture_label"] == arch)
                ]
                if row.empty:
                    raise ValueError(f"Missing result for {topology}, {workload}, {technique}, {arch}.")
                values.append(pd.to_numeric(row[metric], errors="coerce").iloc[0])
            ax.bar(
                [i + (idx - 1) * width for i in x],
                values,
                width=width,
                label=arch,
                color=colors[arch],
                edgecolor="black",
                linewidth=0.4,
            )

        ax.set_title(f"{topology.title()} · {workload.title()} Workload")
        ax.set_xticks(list(x))
        ax.set_xticklabels([technique_labels[name] for name in technique_order], rotation=20, ha="right")
        ax.set_xlabel("Privacy Technique")
        ax.tick_params(axis="both", labelsize=8)
        if metric == "avg_adversary_success_rate":
            ax.set_ylim(0, 1.05)

        axes[topology_idx][0].set_ylabel(ylabel)
    if plot_idx == 0:
        handles, labels = axes[0][0].get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            title="Coordination Mechanisms",
            loc="lower center",
            bbox_to_anchor=(0.5, 1.02),
            ncol=len(arch_order),
            frameon=False,
        )

    fig.tight_layout()
    pdf_path = PLOTS_DIR / filename
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    display(IFrame(src=f"plots/{filename}", width="100%", height=320))

radar_metrics = [
    ("avg_latency_s", "Latency", "min"),
    ("avg_throughput_tasks_per_hour", "Throughput", "max"),
    ("avg_success_rate", "Success\nRate", "max"),
    ("avg_expiration_rate", "Expiration\nRate", "min"),
    ("avg_privacy_added_delay_seconds", "Added\nDelay", "min"),
    ("avg_privacy_processing_seconds", "Processing\nTime", "min"),
    ("avg_adversary_success_rate", "Adversary\nResistance", "min"),
    ("avg_expected_distance_error", "Distance\nError", "max"),
]
radar_ranges = {
    metric: (pd.to_numeric(summary[metric], errors="coerce").min(), pd.to_numeric(summary[metric], errors="coerce").max())
    for metric, _, _ in radar_metrics
}

def radar_score(value, metric, direction):
    minimum, maximum = radar_ranges[metric]
    if pd.isna(value):
        return float("nan")
    if maximum == minimum:
        return 1.0
    score = (float(value) - minimum) / (maximum - minimum)
    return 1.0 - score if direction == "min" else score

for arch in arch_order:
    metrics = radar_metrics if arch != "FD" else radar_metrics[:6]
    angles = [2 * pi * idx / len(metrics) for idx in range(len(metrics))]
    closed_angles = angles + angles[:1]
    fig, axes = plt.subplots(
        len(topology_order),
        len(workload_order),
        figsize=(11, 10),
        subplot_kw={"polar": True},
        squeeze=False,
    )
    for topology, workload in product(topology_order, workload_order):
        topology_idx = topology_order.index(topology)
        workload_idx = workload_order.index(workload)
        ax = axes[topology_idx][workload_idx]
        subset = summary[
            (summary["topology_label"] == topology)
            & (summary["workload"] == workload)
            & (summary["architecture_label"] == arch)
        ]
        for technique in technique_order:
            row = subset[subset["technique"] == technique]
            if row.empty:
                raise ValueError(f"Missing result for {topology}, {workload}, {technique}, {arch}.")
            values = [radar_score(pd.to_numeric(row[metric], errors="coerce").iloc[0], metric, direction) for metric, _, direction in metrics]
            closed_values = values + values[:1]
            color = technique_colors[technique]
            ax.plot(closed_angles, closed_values, color=color, linewidth=1.4, label=technique_labels[technique])
            ax.fill(closed_angles, closed_values, color=color, alpha=0.28)
        ax.set_title(f"{topology.title()} · {workload.title()}", pad=18, fontsize=10)
        ax.set_xticks(angles)
        ax.set_xticklabels([label for _, label, _ in metrics], fontsize=7)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=6, color="#666666")
        ax.grid(color="#BBBBBB", linewidth=0.5, alpha=0.7)
    handles, labels = axes[0][0].get_legend_handles_labels()
    subtitle = "Normalized score · outer is better"
    if arch == "FD":
        subtitle += " · observer metrics N/A because FD has no coordination messages"
    fig.suptitle(f"{arch} · {subtitle}", fontsize=12, y=1.04)
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.0), ncol=len(technique_order), frameon=False)
    fig.tight_layout()
    filename = f"radar_{arch.lower()}_by_technique.pdf"
    fig.savefig(PLOTS_DIR / filename, bbox_inches="tight", transparent=True)
    plt.close(fig)
    display(IFrame(src=f"plots/{filename}", width="100%", height=420))
